In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline,HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [ ]:


video_id = "2VOGakc2qvk" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])
    # print(transcript_list)


    # FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Hi, everyone.', start=0.06, duration=0.66), FetchedTranscriptSnippet(text="My name is Patrick Akil\nand if you're interested in generative AI,", start=0.72, duration=3.0), FetchedTranscriptSnippet(text='large language models\nand some interesting challenges', start=3.74, duration=2.82), FetchedTranscriptSnippet(text="and applications we've seen so far,\nthis episode is for you.", start=6.56, duration=3.5), FetchedTranscriptSnippet(text='Joining me today is Rens Dimmendaal.', start=10.14, duration=1.34), FetchedTranscriptSnippet(text='He is principal data scientist', start=11.48, duration=1.9), FetchedTranscriptSnippet(text='over here at Xebia and trailblazing\nwith the technologies as we speak.', start=13.38, duration=3.68),

    # Convert transcript snippets to single string - use .text attribute, not dictionary indexing
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

In [ ]:
transcript_list

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
len(chunks)

In [37]:
embeddings = HuggingFaceEmbeddings()
vector_store = FAISS.from_documents(chunks, embeddings)

In [39]:
vector_store.index_to_docstore_id

{0: '5f4fa64b-3736-4f11-8df2-d3c12e580fb0',
 1: '2ec13590-1485-4f58-a7f3-2dcb9c42c6dd',
 2: '30ca1b95-20ab-4c00-a506-4be2f9ff32a5',
 3: '206be675-e46e-4453-87d9-3156b1b867b7',
 4: '214a6450-5d5b-4d8e-8a8d-6bf085ae4bb2',
 5: '2965d79c-3848-4d08-a2ad-c59272cd68c3',
 6: 'd938b4f2-c569-4374-be93-51762d6f41b3',
 7: '0ccc07d0-1ce2-4eea-be9f-3320d3f56814',
 8: '1c962093-01b8-4aa3-b1bb-3a5b80fabfb8',
 9: 'a4ba1fdd-a817-4795-94d6-62c5cf2852a0',
 10: '4b37a921-884b-4fb1-aa0d-a5aa6b6c33c8',
 11: '47861bfe-f816-4ce3-84c5-afb2eaf81c11',
 12: 'bec638e1-eed5-4edd-bd3d-9e17dcd93491',
 13: 'abd355e4-b4ec-4162-8593-faf31821a09f',
 14: '8d17b540-2146-4ae9-93df-5f9db6f82ba5',
 15: '43874f3a-f46b-4e6b-9217-ea328cf6bfe8',
 16: '32d9829a-20a8-4ce1-aa8f-9ed0f41a0364',
 17: 'c5e63295-791d-4885-a018-bf0002928a43',
 18: '1ac6a64e-2d5b-42ac-b81f-c9d379a121ac',
 19: '444a80b2-3d84-41e2-9f48-18586f80154c',
 20: '1d8e8182-9be9-45f3-bef0-0181dd8cc698',
 21: '016eb657-6ed8-40e2-949d-38f8a00807a1',
 22: '2dacdc37-7b36-

In [31]:
vector_store.get_by_ids(['332ccab8-bcc8-4c64-9a1f-c00ec099f716'])

[Document(id='332ccab8-bcc8-4c64-9a1f-c00ec099f716', metadata={}, page_content="all I need to use it in the right way. I mean, a hammer in the wrong hands\nis also not so That's not a great idea. Yeah. Yeah, for sure. Cool man. This was, uh. This was a lot of fun. How. How did everything go? I think that was nice.\nThat was good. Yeah. Thanks for coming on that. This is fun. I'm going to round it off here then. Rens Dimmendaal, I'm going to put\nall his socials in the description below. Check them out.\nLet him know you came for our show. And with that being said, thanks for\nlistening. We'll see in the next one.")]

In [40]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x37801a290>, search_kwargs={'k': 4})

In [41]:
retriever.invoke('who is the guest in the video')

[Document(id='5f4fa64b-3736-4f11-8df2-d3c12e580fb0', metadata={}, page_content="Hi, everyone. My name is Patrick Akil\nand if you're interested in generative AI, large language models\nand some interesting challenges and applications we've seen so far,\nthis episode is for you. Joining me today is Rens Dimmendaal. He is principal data scientist over here at Xebia and trailblazing\nwith the technologies as we speak. I’ll put all his socials\nin the description below. Check him out. And with that being said,\nenjoy the episode. Oh, and beyond coding, have you done\nany webinar stuff or something like that? Yeah, I've been here. Like I said, there at the top of the table and then like a customer\nand somebody from my team and I had to like make sure that was nice\nflow of the conversation going on. That was fun. I was it was a good yes for\nI really like speaking or you like, but also I also really like\nbeing the the host there. Yeah. I think that type of role\nto make sure that the flow

In [34]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
llm = HuggingFacePipeline.from_model_id(
    model_id='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    task='text-generation',
    pipeline_kwargs=dict(
        temperature=0.5,
        max_new_tokens=100
    )
)

Device set to use mps:0


In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [42]:
question          = "what was the topics covered in the video"
retrieved_docs    = retriever.invoke(question)

In [43]:
retrieved_docs

[Document(id='28713a81-ab93-47df-ac03-c5740bef5da6', metadata={}, page_content="the topic about learning. I think that will be there. The other part is about when you say\nlike evaluation, when I say that that wisdom access, what we have to do\nis not just about quality control. It's also saying\nwhat should be exactly the way the why like and for example, designing at the figuring out if something is a good use case,\nsomething worth building. Yeah I think that's and then you talk about\ngo back to the to the use cases. I think that today it's easier than before as a single person\nmaybe to make a web products that's something really neat know example auto transcribing YouTube talks. I could go from YouTube video\nand automatically take like from the video all the frames that are actually different\nslides from a conference dog use whisper or something\nto get the transcript out and to automatically go from YouTube\nto blog posts interspersing the slides, plus the transcription\nof wh

In [44]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"the topic about learning. I think that will be there. The other part is about when you say\nlike evaluation, when I say that that wisdom access, what we have to do\nis not just about quality control. It's also saying\nwhat should be exactly the way the why like and for example, designing at the figuring out if something is a good use case,\nsomething worth building. Yeah I think that's and then you talk about\ngo back to the to the use cases. I think that today it's easier than before as a single person\nmaybe to make a web products that's something really neat know example auto transcribing YouTube talks. I could go from YouTube video\nand automatically take like from the video all the frames that are actually different\nslides from a conference dog use whisper or something\nto get the transcript out and to automatically go from YouTube\nto blog posts interspersing the slides, plus the transcription\nof what speaker says into something. Yeah, like seven years, five years ago,\n\nbest

In [45]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      the topic about learning. I think that will be there. The other part is about when you say\nlike evaluation, when I say that that wisdom access, what we have to do\nis not just about quality control. It's also saying\nwhat should be exactly the way the why like and for example, designing at the figuring out if something is a good use case,\nsomething worth building. Yeah I think that's and then you talk about\ngo back to the to the use cases. I think that today it's easier than before as a single person\nmaybe to make a web products that's something really neat know example auto transcribing YouTube talks. I could go from YouTube video\nand automatically take like from the video all the frames that are actually different\nslides from a conference dog use whisper or something\nto get the transcript 

In [47]:
answer = llm.invoke(final_prompt)
print(answer)


      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      the topic about learning. I think that will be there. The other part is about when you say
like evaluation, when I say that that wisdom access, what we have to do
is not just about quality control. It's also saying
what should be exactly the way the why like and for example, designing at the figuring out if something is a good use case,
something worth building. Yeah I think that's and then you talk about
go back to the to the use cases. I think that today it's easier than before as a single person
maybe to make a web products that's something really neat know example auto transcribing YouTube talks. I could go from YouTube video
and automatically take like from the video all the frames that are actually different
slides from a conference dog use whisper or something
to get the transcript out and to automatically go from YouTu

# Building a Chain


In [48]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [49]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})


In [52]:
parallel_chain.invoke('who is the guest in the video') 

{'context': "Hi, everyone. My name is Patrick Akil\nand if you're interested in generative AI, large language models\nand some interesting challenges and applications we've seen so far,\nthis episode is for you. Joining me today is Rens Dimmendaal. He is principal data scientist over here at Xebia and trailblazing\nwith the technologies as we speak. I’ll put all his socials\nin the description below. Check him out. And with that being said,\nenjoy the episode. Oh, and beyond coding, have you done\nany webinar stuff or something like that? Yeah, I've been here. Like I said, there at the top of the table and then like a customer\nand somebody from my team and I had to like make sure that was nice\nflow of the conversation going on. That was fun. I was it was a good yes for\nI really like speaking or you like, but also I also really like\nbeing the the host there. Yeah. I think that type of role\nto make sure that the flow is right. Yeah, I agree. It's the first time I went on a podcast\n

In [53]:
parser = StrOutputParser()

In [54]:
main_chain = parallel_chain | prompt | llm | parser

In [55]:
main_chain.invoke('Can you summarize the video')

"\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      a problem. We just show something cool. Yeah. And that is easier than I think it's ever\nbeen. What did you make? I so I suck at writing summaries\nfor my podcast and I've stopped doing it. I just do the outline, which\nis the timestamps and what we talk about. So my idea was on an innovation day. Okay,\nif I have my audio file from the podcast, I want to give that to whisper and it's\ngoing to give me the transcribed version and I'm going to feed that into chat\nand I'm going to be okay. This is the podcast, this is the context. Can you write me a summary? And it did the thing. It just took a long time. Also, I didn't want to pay, so I had like a limited amount\nof characters and stuff like that. But this was early on\nand a summary came out and I was like, This is actually fairly doable. Like if you tweak this or if yo